## Lecture-03 Walkthrough Example - Bigram Language Model 


### Step 1 Add padding

In [22]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.util import bigrams
from prettytable import PrettyTable
from collections import Counter, defaultdict
import random

# Sample text
sample_text = "I like NLP. NLP is fun. NLP in python is fun."

# Tokenize the sample_text into sentences
sentences = sent_tokenize(sample_text)

# Pad each sentence individually with <s> and </s> tokens and tokenize into words
padded_sentences = []
for sentence in sentences:
    # Tokenize sentence into words
    words = word_tokenize(sentence)
    padded = ['<s>'] + words + ['</s>']
    padded_sentences.append(padded)

print("Padded Sentences:")
for sent in padded_sentences:
    print(sent)

Padded Sentences:
['<s>', 'I', 'like', 'NLP', '.', '</s>']
['<s>', 'NLP', 'is', 'fun', '.', '</s>']
['<s>', 'NLP', 'in', 'python', 'is', 'fun', '.', '</s>']



### Step 2 Create Unigram and Bigram list


In [23]:
# Create bigrams for each padded sentence separately
all_bigrams = []
for sent in padded_sentences:
    sentence_bigrams = list(bigrams(sent))
    all_bigrams.extend(sentence_bigrams)

# Print all bigrams using PrettyTable
btab = PrettyTable(["Index", "Bigram"])
for i, bg in enumerate(all_bigrams, start=1):
    btab.add_row([i, bg])
print("All Bigrams:")
print(btab)

# Also, collect and print all unigrams (tokens) from all sentences
all_unigrams = [token for sent in padded_sentences for token in sent]
utab = PrettyTable(["Index", "Unigram"])
for i, token in enumerate(all_unigrams, start=1):
    utab.add_row([i, token])
print("All Unigrams:")
print(utab)

All Bigrams:
+-------+------------------+
| Index |      Bigram      |
+-------+------------------+
|   1   |   ('<s>', 'I')   |
|   2   |  ('I', 'like')   |
|   3   | ('like', 'NLP')  |
|   4   |   ('NLP', '.')   |
|   5   |  ('.', '</s>')   |
|   6   |  ('<s>', 'NLP')  |
|   7   |  ('NLP', 'is')   |
|   8   |  ('is', 'fun')   |
|   9   |   ('fun', '.')   |
|   10  |  ('.', '</s>')   |
|   11  |  ('<s>', 'NLP')  |
|   12  |  ('NLP', 'in')   |
|   13  | ('in', 'python') |
|   14  | ('python', 'is') |
|   15  |  ('is', 'fun')   |
|   16  |   ('fun', '.')   |
|   17  |  ('.', '</s>')   |
+-------+------------------+
All Unigrams:
+-------+---------+
| Index | Unigram |
+-------+---------+
|   1   |   <s>   |
|   2   |    I    |
|   3   |   like  |
|   4   |   NLP   |
|   5   |    .    |
|   6   |   </s>  |
|   7   |   <s>   |
|   8   |   NLP   |
|   9   |    is   |
|   10  |   fun   |
|   11  |    .    |
|   12  |   </s>  |
|   13  |   <s>   |
|   14  |   NLP   |
|   15  |    in   |
|   

### Step 3 Count the occurrence of unique Unigram and Bigram 

In [24]:
# Calculate frequency counts of bigrams and unigrams
bigram_counts = Counter(all_bigrams)
unigram_counts = Counter(all_unigrams)

c_bi_tab = PrettyTable(["Index", "Unique Bigram", "Count"])
for i, (bg, count) in enumerate(bigram_counts.items(), start=1):
    c_bi_tab.add_row([i, bg, count])
print("Bigram Counts:")
print(c_bi_tab)

c_uni_tab = PrettyTable(["Index", "Unique Unigram", "Count"])
for i, (token, count) in enumerate(unigram_counts.items(), start=1):
    c_uni_tab.add_row([i, token, count])
print("Unigram Counts:")
print(c_uni_tab)

Bigram Counts:
+-------+------------------+-------+
| Index |  Unique Bigram   | Count |
+-------+------------------+-------+
|   1   |   ('<s>', 'I')   |   1   |
|   2   |  ('I', 'like')   |   1   |
|   3   | ('like', 'NLP')  |   1   |
|   4   |   ('NLP', '.')   |   1   |
|   5   |  ('.', '</s>')   |   3   |
|   6   |  ('<s>', 'NLP')  |   2   |
|   7   |  ('NLP', 'is')   |   1   |
|   8   |  ('is', 'fun')   |   2   |
|   9   |   ('fun', '.')   |   2   |
|   10  |  ('NLP', 'in')   |   1   |
|   11  | ('in', 'python') |   1   |
|   12  | ('python', 'is') |   1   |
+-------+------------------+-------+
Unigram Counts:
+-------+----------------+-------+
| Index | Unique Unigram | Count |
+-------+----------------+-------+
|   1   |      <s>       |   3   |
|   2   |       I        |   1   |
|   3   |      like      |   1   |
|   4   |      NLP       |   3   |
|   5   |       .        |   3   |
|   6   |      </s>      |   3   |
|   7   |       is       |   2   |
|   8   |      fun       | 

### Step 4 Calculate the MLE probabilities for Bigram 

In [25]:
# Calculate the MLE for each bigram (conditional probability)
bi_mle = defaultdict(dict)
for (w1, w2), count in bigram_counts.items():
    bi_mle[w1][w2] = round(count / unigram_counts[w1], 2)

# Display the MLEs in a PrettyTable
mle_table = PrettyTable(["Word 1", "Word 2", "MLE of Bigram"])
for w1, following in bi_mle.items():
    for w2, mle in following.items():
        mle_table.add_row([w1, w2, mle])
print("Bigram MLEs:")
print(mle_table)

Bigram MLEs:
+--------+--------+---------------+
| Word 1 | Word 2 | MLE of Bigram |
+--------+--------+---------------+
|  <s>   |   I    |      0.33     |
|  <s>   |  NLP   |      0.67     |
|   I    |  like  |      1.0      |
|  like  |  NLP   |      1.0      |
|  NLP   |   .    |      0.33     |
|  NLP   |   is   |      0.33     |
|  NLP   |   in   |      0.33     |
|   .    |  </s>  |      1.0      |
|   is   |  fun   |      1.0      |
|  fun   |   .    |      1.0      |
|   in   | python |      1.0      |
| python |   is   |      1.0      |
+--------+--------+---------------+


##  Next word prediction in above builded bigram language model


In [26]:
import random

# Define a function that predict the next word given the input word based on the highest probability.

def predict_next(w_input, mle):
    prob_w_next = mle.get(w_input, {})
    # If there are no next words, return None
    if not prob_w_next:
        return None
    else:
    # Return the word with the highest probability
        return max(prob_w_next, key= prob_w_next.get)

word = input('Type word for which you want to predict the next word: ')
predicted_word = predict_next(word, bi_mle)
print(f"Predicted next word after {word}: {predicted_word}")

Type word for which you want to predict the next word: is
Predicted next word after is: fun


## Bonus: Sentence generation

In [27]:
import random

def generate_process(model, max_length):
    current_word = random.choice(list(model.keys()))  # Choose a random initial word from the model
    get_text = [current_word]  # Initialize a list to store the generated text
    for _ in range(max_length - 1):  # Iterate up to max_length - 1 to prevent infinite loops
        w_next_prob = model.get(get_text[-1], {})  # Get the probabilities of next words given the current word
        if not w_next_prob:  # If there are no probabilities for the current word, break the loop
            break
        w_next = random.choices(list(w_next_prob.keys()), weights=list(w_next_prob.values()))[0]  # Choose the next word randomly based on its probabilities
        get_text.append(w_next)  # Append the next word to the generated text
    return ' '.join(get_text)  # Return the generated text as a string

# Example usage
generated_text=generate_process(bi_mle, 4)
print(f"Generated text: {generated_text}")

Generated text: fun . </s>


## Bonus: Laplace Smoothing 

In [32]:
# Vocabulary Size
V = len(unigram_counts)
lap_mle = defaultdict(dict)

for (w1, w2), c in bigram_counts.items():
    lap_mle[w1][w2]=round((c+1)/(unigram_counts[w1]+V),2)

table = PrettyTable(['Word 1 in Bigram ', 'Word 2 in Bigram ',' Laplace MLE of Bigram'])
for key, value in lap_mle.items():
    table.add_row([key,[k for k,v in value.items()],value])
print(table)

+-------------------+-------------------+-------------------------------------+
| Word 1 in Bigram  | Word 2 in Bigram  |         Laplace MLE of Bigram       |
+-------------------+-------------------+-------------------------------------+
|        <s>        |    ['I', 'NLP']   |       {'I': 0.15, 'NLP': 0.23}      |
|         I         |      ['like']     |            {'like': 0.18}           |
|        like       |      ['NLP']      |            {'NLP': 0.18}            |
|        NLP        | ['.', 'is', 'in'] | {'.': 0.15, 'is': 0.15, 'in': 0.15} |
|         .         |      ['</s>']     |            {'</s>': 0.31}           |
|         is        |      ['fun']      |            {'fun': 0.25}            |
|        fun        |       ['.']       |             {'.': 0.25}             |
|         in        |     ['python']    |           {'python': 0.18}          |
|       python      |       ['is']      |             {'is': 0.18}            |
+-------------------+-------------------

## Bonus: Perplexity

In [33]:
from prettytable import PrettyTable

# Define the list of bigrams and their corresponding probabilities
bigram_list = [("<s>", "I"), ("I", "like"), ("like", "NLP"), ("NLP", "</s>")]
probs = [0.5, 0.4, 0.3, 0.2]


# Create a PrettyTable to display each bigram with its probability
table = PrettyTable(["Bigram", "Probability"])
for bigram, prob in zip(bigram_list, probs):
    # Format the bigram for display
    table.add_row([f"{bigram[0]} {bigram[1]}", prob])
    
print("Bigram Probabilities:")
print(table)


# Calculate the product of all probabilities
prod = 1
for p in probs:
    prod *= p

# Number of transitions (bigram probabilities)
N = len(probs)

# Calculate perplexity: nth root of the reciprocal of the product
perplexity = (1 / prod) ** (1 / N)
print("Perplexity:", perplexity)


Bigram Probabilities:
+----------+-------------+
|  Bigram  | Probability |
+----------+-------------+
|  <s> I   |     0.5     |
|  I like  |     0.4     |
| like NLP |     0.3     |
| NLP </s> |     0.2     |
+----------+-------------+
Perplexity: 3.0213753973567683


## Bonus: Bigram Model with NLTK abc Corpus

In [34]:
#Download the abc corpus 
#nltk.download('abc')
#nltk.download('punkt')

import nltk
from nltk.corpus import abc
from nltk import bigrams
from collections import Counter, defaultdict


# Step 1: Get sentences from the ABC corpus
sentences = abc.sents()[:50]
print("Number of sentences used:", len(sentences))

# Step 2: For each sentence, add start (<s>) and end (</s>) tokens,
# then combine all padded sentences into a single list of tokens.
tokens = []
for sentence in sentences:
    padded_sentence = ['<s>'] + sentence + ['</s>']
    tokens.extend(padded_sentence)
print("Example tokens:", tokens[:10])  # Print the first 20 tokens for inspection

# Step 3: Create bigrams from the list of tokens
bigram_list = list(bigrams(tokens))
print("Example bigrams:", bigram_list[:10])

# Step 4: Count the frequency of each bigram and each individual word (unigram)
bigram_counts = Counter(bigram_list)
unigram_counts = Counter(tokens)

# Step 5: Calculate the Maximum Likelihood Estimate (MLE) for each bigram.
# For each bigram (w1, w2), the probability is count(w1, w2) / count(w1).
bi_mle = defaultdict(dict)
for (w1, w2), count in bigram_counts.items():
    probability = count / unigram_counts[w1]
    bi_mle[w1][w2] = round(probability, 2)

# Function to predict the next word based on the highest conditional probability for a given word.
def predict_next(word, mle):
    next_words = mle.get(word, {})
    return max(next_words, key=next_words.get) if next_words else None

# Step 6: Ask the user for a word and predict the next word from the corpus
input_word = input("Enter a word from the corpus to predict its next word: ")
predicted_word = predict_next(input_word, bi_mle)
print(f"Predicted next word after '{input_word}': {predicted_word}")


Number of sentences used: 50
Example tokens: ['<s>', 'PM', 'denies', 'knowledge', 'of', 'AWB', 'kickbacks', 'The', 'Prime', 'Minister']
Example bigrams: [('<s>', 'PM'), ('PM', 'denies'), ('denies', 'knowledge'), ('knowledge', 'of'), ('of', 'AWB'), ('AWB', 'kickbacks'), ('kickbacks', 'The'), ('The', 'Prime'), ('Prime', 'Minister'), ('Minister', 'has')]
Enter a word from the corpus to predict its next word: of
Predicted next word after 'of': the
